<a href="https://colab.research.google.com/github/ssykes-eth/ETH_273-0003-00L/blob/weekend_2/diffusion_theory/gaussian_visualizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# The Shape of Randomness
An interactive Gaussian sampling visualizer — no build step required.

Run the cell below. The interactive app will appear inline.

In [1]:
#@title Interactive Visualizer { display-mode: "form" }
from IPython.display import HTML

html = """
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8" />
<style>
  @import url('https://fonts.googleapis.com/css2?family=DM+Serif+Display:ital@0;1&family=DM+Sans:wght@300;400;500;600&display=swap');
  * { box-sizing: border-box; margin: 0; padding: 0; }
  body { background: #FDFBF7; font-family: 'DM Sans', sans-serif; }
</style>
</head>
<body>
<div id="root"></div>

<!-- Dependencies -->
<script src="https://unpkg.com/react@18/umd/react.production.min.js"></script>
<script src="https://unpkg.com/react-dom@18/umd/react-dom.production.min.js"></script>
<script src="https://unpkg.com/@babel/standalone/babel.min.js"></script>
<script src="https://unpkg.com/d3@7/dist/d3.min.js"></script>

<script type="text/babel">
const { useState, useEffect, useRef, useCallback, useMemo } = React;

function uniformSample(lo, hi) {
  return lo + Math.random() * (hi - lo);
}

function gaussianPDF(x, sigma) {
  const s2 = sigma * sigma;
  return (1 / Math.sqrt(2 * Math.PI * s2)) * Math.exp(-(x * x) / (2 * s2));
}

function generatePanel1Samples(n, count = 500) {
  return Array.from({ length: count }, () => {
    let s = 0;
    for (let i = 0; i < n; i++) s += uniformSample(-1, 1);
    return s / Math.sqrt(n / 3);
  });
}

function generatePanel2Samples(spread, count = 500) {
  const n = 20;
  return Array.from({ length: count }, () => {
    let s = 0;
    for (let i = 0; i < n; i++) s += uniformSample(-spread, spread);
    return s / Math.sqrt(n / 3);
  });
}

function generatePanel3Samples(spreadA, spreadB, count = 500) {
  const n = 20;
  const samplesA = Array.from({ length: count }, () => {
    let s = 0;
    for (let i = 0; i < n; i++) s += uniformSample(-spreadA, spreadA);
    return s / Math.sqrt(n / 3);
  });
  const samplesB = Array.from({ length: count }, () => {
    let s = 0;
    for (let i = 0; i < n; i++) s += uniformSample(-spreadB, spreadB);
    return s / Math.sqrt(n / 3);
  });
  return { samplesA, samplesB, samplesSum: samplesA.map((a, i) => a + samplesB[i]) };
}

const NUM_BINS = 45;
const X_RANGE = [-6, 6];

function buildBins(samples) {
  const thresholds = d3.range(NUM_BINS + 1).map(
    i => X_RANGE[0] + i * (X_RANGE[1] - X_RANGE[0]) / NUM_BINS
  );
  return d3.bin().domain(X_RANGE).thresholds(thresholds)(samples);
}

function Histogram({ samples, sigma, color, width = 520, height = 220, showSpreadShading, spread }) {
  const svgRef = useRef(null);
  const bins = useMemo(() => buildBins(samples), [samples]);

  useEffect(() => {
    if (!svgRef.current) return;
    const margin = { top: 16, right: 16, bottom: 32, left: 40 };
    const W = width - margin.left - margin.right;
    const H = height - margin.top - margin.bottom;
    const svg = d3.select(svgRef.current);
    svg.selectAll('*').remove();
    const g = svg.append('g').attr('transform', `translate(${margin.left},${margin.top})`);
    const xScale = d3.scaleLinear().domain(X_RANGE).range([0, W]);
    const binWidth = xScale(bins[0]?.x1 ?? 1) - xScale(bins[0]?.x0 ?? 0);
    const totalCount = samples.length || 1;
    const binWidthVal = (X_RANGE[1] - X_RANGE[0]) / NUM_BINS;
    const maxDensity = d3.max(bins, d => d.length / (totalCount * binWidthVal)) || 1;
    const yMax = Math.max(maxDensity, gaussianPDF(0, sigma)) * 1.15;
    const yScale = d3.scaleLinear().domain([0, yMax]).range([H, 0]);

    if (showSpreadShading && spread) {
      g.append('rect')
        .attr('x', xScale(-spread)).attr('width', xScale(spread) - xScale(-spread))
        .attr('y', 0).attr('height', H).attr('fill', color).attr('opacity', 0.08);
    }

    g.selectAll('.bar').data(bins).enter().append('rect')
      .attr('x', d => xScale(d.x0) + 0.5).attr('width', Math.max(0, binWidth - 1))
      .attr('y', H).attr('height', 0).attr('fill', color).attr('opacity', 0.65).attr('rx', 1)
      .transition().duration(300)
      .attr('y', d => yScale(d.length / (totalCount * binWidthVal)))
      .attr('height', d => H - yScale(d.length / (totalCount * binWidthVal)));

    const curvePoints = d3.range(200).map(i => {
      const x = X_RANGE[0] + i * (X_RANGE[1] - X_RANGE[0]) / 199;
      return [xScale(x), yScale(gaussianPDF(x, sigma))];
    });
    g.append('path').datum(curvePoints)
      .attr('d', d3.line().x(d => d[0]).y(d => d[1]).curve(d3.curveCatmullRom))
      .attr('fill', 'none').attr('stroke', color).attr('stroke-width', 2.5).attr('opacity', 0.7);

    g.append('g').attr('transform', `translate(0,${H})`)
      .call(d3.axisBottom(xScale).ticks(7).tickSize(3))
      .call(ax => ax.select('.domain').attr('stroke', '#aaa'))
      .call(ax => ax.selectAll('.tick text').style('font-size', '10px').attr('fill', '#666'));
    g.append('g').call(d3.axisLeft(yScale).ticks(4).tickSize(3))
      .call(ax => ax.select('.domain').attr('stroke', '#aaa'))
      .call(ax => ax.selectAll('.tick text').style('font-size', '10px').attr('fill', '#666'));
    g.append('text').attr('x', W / 2).attr('y', H + 28)
      .attr('text-anchor', 'middle').style('font-size', '11px').attr('fill', '#888').text('value');
  }, [samples, sigma, color, width, height, bins, showSpreadShading, spread]);

  return <svg ref={svgRef} width={width} height={height} style={{ display: 'block' }} />;
}

function PythagoreanTriangle({ a, b }) {
  const SCALE = 45;
  const legA = a * SCALE, legB = b * SCALE;
  const hyp = Math.sqrt(a * a + b * b);
  const pad = 32;
  const W = legB + pad * 2 + 40, H = legA + pad * 2 + 20;
  const x0 = pad, y0 = pad + legA, x1 = pad, y1 = pad, x2 = pad + legB, y2 = pad + legA;
  return (
    <svg width={W} height={H} style={{ display: 'block', margin: '0 auto' }}>
      <polygon points={`${x0},${y0} ${x1},${y1} ${x2},${y2}`} fill="#e8dff5" stroke="#7B5EA7" strokeWidth="2" />
      <polyline points={`${x0+10},${y0} ${x0+10},${y0-10} ${x0},${y0-10}`} fill="none" stroke="#7B5EA7" strokeWidth="1.5" />
      <text x={x0-12} y={(y0+y1)/2} textAnchor="middle" dominantBaseline="middle" fill="#E07A5F" fontSize="13" fontWeight="600">A={a.toFixed(1)}</text>
      <text x={(x0+x2)/2} y={y2+18} textAnchor="middle" dominantBaseline="middle" fill="#3D85C6" fontSize="13" fontWeight="600">B={b.toFixed(1)}</text>
      <text x={(x1+x2)/2+14} y={(y1+y2)/2-8} textAnchor="middle" dominantBaseline="middle" fill="#7B5EA7" fontSize="13" fontWeight="600">
        \u221a(A\u00b2+B\u00b2)\u2248{hyp.toFixed(2)}
      </text>
    </svg>
  );
}

function Panel1() {
  const [n, setN] = useState(1);
  const [samples, setSamples] = useState([]);
  const [animating, setAnimating] = useState(false);

  const caption = useMemo(() => {
    if (n === 1) return "With just 1 draw, every value is equally likely \u2014 a flat shape.";
    if (n <= 3) return "Already starting to pile up in the middle...";
    if (n <= 9) return "Looking more and more like a bell!";
    return "That\u2019s a bell curve! This is what a Gaussian distribution looks like.";
  }, [n]);

  const handleDraw = useCallback(() => {
    if (animating) return;
    setAnimating(true); setSamples([]);
    const all = generatePanel1Samples(n, 500);
    let batch = 0;
    const timer = setInterval(() => {
      batch++; setSamples(all.slice(0, batch * 50));
      if (batch >= 10) { clearInterval(timer); setAnimating(false); }
    }, 90);
  }, [n, animating]);

  const btn = { background: animating ? '#ccc' : '#E07A5F', color: 'white', border: 'none', borderRadius: 8, padding: '10px 24px', fontSize: '1rem', fontWeight: 600, cursor: animating ? 'not-allowed' : 'pointer', marginBottom: '1.5rem' };
  return (
    <div>
      <h2 style={{ fontFamily: 'DM Serif Display, serif', fontSize: '1.5rem', color: '#3a2a1e', marginBottom: 4 }}>What is a Bell Curve?</h2>
      <p style={{ color: '#666', marginBottom: '1rem', maxWidth: 560, lineHeight: 1.6 }}>
        If you add up many small random numbers, you always get a bell-shaped result. Try it!
      </p>
      <div style={{ display: 'flex', alignItems: 'center', gap: '1rem', marginBottom: '1rem', flexWrap: 'wrap' }}>
        <label style={{ fontWeight: 500, color: '#3a2a1e' }}>
          How many random numbers to add up: <span style={{ color: '#E07A5F', fontWeight: 700 }}>{n}</span>
        </label>
        <input type="range" min={1} max={30} value={n}
          onChange={e => { setN(Number(e.target.value)); setSamples([]); }}
          style={{ width: 180, accentColor: '#E07A5F' }} />
      </div>
      <button onClick={handleDraw} disabled={animating} style={btn}>{animating ? 'Drawing...' : 'Draw 500 samples'}</button>
      <div style={{ background: 'white', borderRadius: 12, boxShadow: '0 2px 12px rgba(0,0,0,0.07)', padding: '1rem', display: 'inline-block' }}>
        <Histogram samples={samples} sigma={1} color="#E07A5F" width={520} height={220} />
      </div>
      <div style={{ marginTop: '1rem', padding: '0.75rem 1rem', background: '#fff8f5', borderLeft: '3px solid #E07A5F', borderRadius: 4, maxWidth: 520, color: '#3a2a1e', fontStyle: 'italic', lineHeight: 1.6 }}>
        {caption}
      </div>
    </div>
  );
}

function Panel2() {
  const [spread, setSpread] = useState(1.0);
  const [samples, setSamples] = useState([]);
  const [animating, setAnimating] = useState(false);

  const handleDraw = useCallback(() => {
    if (animating) return;
    setAnimating(true); setSamples([]);
    const all = generatePanel2Samples(spread, 500);
    let batch = 0;
    const timer = setInterval(() => {
      batch++; setSamples(all.slice(0, batch * 50));
      if (batch >= 10) { clearInterval(timer); setAnimating(false); }
    }, 90);
  }, [spread, animating]);

  const btn = { background: animating ? '#ccc' : '#E07A5F', color: 'white', border: 'none', borderRadius: 8, padding: '10px 24px', fontSize: '1rem', fontWeight: 600, cursor: animating ? 'not-allowed' : 'pointer', marginBottom: '1.5rem' };
  return (
    <div>
      <h2 style={{ fontFamily: 'DM Serif Display, serif', fontSize: '1.5rem', color: '#3a2a1e', marginBottom: 4 }}>Changing the Spread</h2>
      <p style={{ color: '#666', marginBottom: '1rem', maxWidth: 560, lineHeight: 1.6 }}>
        Every bell curve has a \u201cspread\u201d \u2014 how wide or narrow it is. Drag the slider to see.
      </p>
      <div style={{ display: 'flex', alignItems: 'center', gap: '1rem', marginBottom: '1rem', flexWrap: 'wrap' }}>
        <label style={{ fontWeight: 500, color: '#3a2a1e' }}>
          Spread: <span style={{ color: '#E07A5F', fontWeight: 700 }}>{spread.toFixed(1)}</span>
        </label>
        <input type="range" min={0.5} max={3.0} step={0.1} value={spread}
          onChange={e => { setSpread(Number(e.target.value)); setSamples([]); }}
          style={{ width: 200, accentColor: '#E07A5F' }} />
      </div>
      <button onClick={handleDraw} disabled={animating} style={btn}>{animating ? 'Drawing...' : 'Draw 500 samples'}</button>
      <div style={{ background: 'white', borderRadius: 12, boxShadow: '0 2px 12px rgba(0,0,0,0.07)', padding: '1rem', display: 'inline-block' }}>
        <Histogram samples={samples} sigma={spread} color="#E07A5F" width={520} height={220} showSpreadShading spread={spread} />
      </div>
      <div style={{ marginTop: '1rem', padding: '0.75rem 1rem', background: '#fff8f5', borderLeft: '3px solid #E07A5F', borderRadius: 4, maxWidth: 520, color: '#3a2a1e', lineHeight: 1.6 }}>
        A <strong>wider spread</strong> means the bell curve is wider and flatter.
        A <strong>narrower spread</strong> means values cluster more tightly around zero.
      </div>
    </div>
  );
}

function Panel3() {
  const [spreadA, setSpreadA] = useState(1.0);
  const [spreadB, setSpreadB] = useState(1.0);
  const [samplesA, setSamplesA] = useState([]);
  const [samplesB, setSamplesB] = useState([]);
  const [samplesSum, setSamplesSum] = useState([]);
  const [animating, setAnimating] = useState(false);
  const [showInsight, setShowInsight] = useState(false);

  const predicted = Math.sqrt(spreadA * spreadA + spreadB * spreadB);

  const handleDraw = useCallback(() => {
    if (animating) return;
    setAnimating(true); setSamplesA([]); setSamplesB([]); setSamplesSum([]);
    const { samplesA: allA, samplesB: allB, samplesSum: allSum } = generatePanel3Samples(spreadA, spreadB, 500);
    let batch = 0;
    const timer = setInterval(() => {
      batch++;
      setSamplesA(allA.slice(0, batch * 50));
      setSamplesB(allB.slice(0, batch * 50));
      setSamplesSum(allSum.slice(0, batch * 50));
      if (batch >= 10) { clearInterval(timer); setAnimating(false); }
    }, 90);
  }, [spreadA, spreadB, animating]);

  const btn = { background: animating ? '#ccc' : '#7B5EA7', color: 'white', border: 'none', borderRadius: 8, padding: '10px 24px', fontSize: '1rem', fontWeight: 600, cursor: animating ? 'not-allowed' : 'pointer', marginBottom: '1.5rem' };
  const histW = 340, histH = 180;

  return (
    <div>
      <h2 style={{ fontFamily: 'DM Serif Display, serif', fontSize: '1.5rem', color: '#3a2a1e', marginBottom: 4 }}>Adding Two Bell Curves</h2>
      <p style={{ color: '#666', marginBottom: '1rem', maxWidth: 640, lineHeight: 1.6 }}>
        Add a sample from one bell curve to a sample from another. The result is a new bell curve \u2014 whose spread follows the Pythagorean theorem!
      </p>
      <div style={{ display: 'flex', gap: '2rem', flexWrap: 'wrap', marginBottom: '1rem' }}>
        <div style={{ display: 'flex', alignItems: 'center', gap: '0.75rem' }}>
          <label style={{ fontWeight: 500, color: '#3a2a1e' }}>
            <span style={{ color: '#E07A5F', fontWeight: 700 }}>Spread A:</span>{' '}
            <span style={{ color: '#E07A5F', fontWeight: 700 }}>{spreadA.toFixed(1)}</span>
          </label>
          <input type="range" min={0.5} max={3.0} step={0.1} value={spreadA}
            onChange={e => { setSpreadA(Number(e.target.value)); setSamplesA([]); setSamplesSum([]); }}
            style={{ width: 140, accentColor: '#E07A5F' }} />
        </div>
        <div style={{ display: 'flex', alignItems: 'center', gap: '0.75rem' }}>
          <label style={{ fontWeight: 500, color: '#3a2a1e' }}>
            <span style={{ color: '#3D85C6', fontWeight: 700 }}>Spread B:</span>{' '}
            <span style={{ color: '#3D85C6', fontWeight: 700 }}>{spreadB.toFixed(1)}</span>
          </label>
          <input type="range" min={0.5} max={3.0} step={0.1} value={spreadB}
            onChange={e => { setSpreadB(Number(e.target.value)); setSamplesB([]); setSamplesSum([]); }}
            style={{ width: 140, accentColor: '#3D85C6' }} />
        </div>
      </div>
      <button onClick={handleDraw} disabled={animating} style={btn}>{animating ? 'Drawing...' : 'Draw 500 samples'}</button>
      <div style={{ display: 'flex', gap: '1rem', flexWrap: 'wrap', marginBottom: '1.5rem' }}>
        {[['Bell Curve A', '#E07A5F', samplesA, spreadA], ['Bell Curve B', '#3D85C6', samplesB, spreadB], ['A + B', '#7B5EA7', samplesSum, predicted]]
          .map(([label, color, data, sig]) => (
            <div key={label}>
              <div style={{ fontSize: '0.8rem', fontWeight: 600, color, marginBottom: 4, textAlign: 'center' }}>{label}</div>
              <div style={{ background: 'white', borderRadius: 10, boxShadow: '0 2px 10px rgba(0,0,0,0.07)', padding: '0.5rem' }}>
                <Histogram samples={data} sigma={sig} color={color} width={histW} height={histH} />
              </div>
            </div>
          ))}
      </div>
      <div style={{ background: 'white', borderRadius: 12, boxShadow: '0 2px 12px rgba(0,0,0,0.07)', padding: '1rem 1.25rem', maxWidth: 480, marginBottom: '1rem' }}>
        <div style={{ fontFamily: 'DM Serif Display, serif', fontSize: '1rem', color: '#3a2a1e', marginBottom: '0.5rem' }}>The Pythagorean Theorem of Spread</div>
        <PythagoreanTriangle a={spreadA} b={spreadB} />
        <div style={{ textAlign: 'center', marginTop: 8, color: '#555', fontSize: '0.9rem', lineHeight: 1.5 }}>
          <strong style={{ color: '#E07A5F' }}>A\u00b2</strong> + <strong style={{ color: '#3D85C6' }}>B\u00b2</strong> = <strong style={{ color: '#7B5EA7' }}>({predicted.toFixed(2)})\u00b2</strong><br />
          <span style={{ fontSize: '0.8rem', color: '#888' }}>
            {spreadA.toFixed(1)}\u00b2 + {spreadB.toFixed(1)}\u00b2 = {(spreadA*spreadA + spreadB*spreadB).toFixed(2)} \u2192 spread = {predicted.toFixed(2)}
          </span>
        </div>
      </div>
      <div style={{ padding: '0.75rem 1rem', background: '#f3effe', borderLeft: '3px solid #7B5EA7', borderRadius: 4, maxWidth: 540, color: '#3a2a1e', lineHeight: 1.6, marginBottom: '0.75rem' }}>
        <strong>The spread of the sum follows the Pythagorean theorem!</strong> Just like the hypotenuse, the combined spread is \u221a(A\u00b2 + B\u00b2).
      </div>
      <button onClick={() => setShowInsight(v => !v)}
        style={{ background: 'transparent', border: '1.5px solid #7B5EA7', color: '#7B5EA7', borderRadius: 8, padding: '7px 18px', fontSize: '0.9rem', fontWeight: 500, cursor: 'pointer' }}>
        {showInsight ? 'Hide insight' : 'Why does this work? \u25be'}
      </button>
      {showInsight && (
        <div style={{ marginTop: '0.75rem', padding: '1rem 1.25rem', background: '#faf7ff', border: '1px solid #d5c8f0', borderRadius: 8, maxWidth: 540, color: '#3a2a1e', lineHeight: 1.7, fontSize: '0.9rem' }}>
          Each sample of A is built from 20 small random draws. Each sample of B from 20 others.
          When we add them we\u2019re really adding <em>pairs</em>. The combined spread of each pair is A\u00b2 + B\u00b2,
          so the total spread is \u221a(A\u00b2 + B\u00b2). That\u2019s the Pythagorean theorem!
        </div>
      )}
    </div>
  );
}

const TABS = [
  { id: 'p1', label: '1. What is a Bell Curve?' },
  { id: 'p2', label: '2. Changing the Spread' },
  { id: 'p3', label: '3. Adding Two Bell Curves' },
];

function App() {
  const [tab, setTab] = useState('p1');
  return (
    <div style={{ minHeight: '100vh', background: '#FDFBF7', fontFamily: 'DM Sans, sans-serif' }}>
      <div style={{ background: 'white', borderBottom: '1px solid #e8e2d9', padding: '1.25rem 1.5rem 0' }}>
        <h1 style={{ fontFamily: 'DM Serif Display, serif', fontSize: '1.8rem', color: '#2a1e14', marginBottom: 2 }}>The Shape of Randomness</h1>
        <p style={{ color: '#888', marginBottom: '1rem', fontSize: '0.9rem' }}>An interactive tour of bell curves \u2014 no math required.</p>
        <div style={{ display: 'flex' }}>
          {TABS.map(t => (
            <button key={t.id} onClick={() => setTab(t.id)}
              style={{ background: 'transparent', border: 'none', borderBottom: tab === t.id ? '3px solid #E07A5F' : '3px solid transparent', color: tab === t.id ? '#E07A5F' : '#888', fontWeight: tab === t.id ? 600 : 400, fontSize: '0.9rem', padding: '0.75rem 1rem', cursor: 'pointer', whiteSpace: 'nowrap', fontFamily: 'DM Sans, sans-serif' }}>
              {t.label}
            </button>
          ))}
        </div>
      </div>
      <div style={{ padding: '1.5rem' }}>
        {tab === 'p1' && <Panel1 />}
        {tab === 'p2' && <Panel2 />}
        {tab === 'p3' && <Panel3 />}
      </div>
    </div>
  );
}

ReactDOM.createRoot(document.getElementById('root')).render(<App />);
</script>
</body>
</html>
"""

display(HTML(f'<iframe srcdoc="{html.replace(chr(34), "&quot;").replace(chr(39), "&#39;")}" width="100%" height="900" style="border:none;"></iframe>'))


/usr/local/lib/python3.12/dist-packages/IPython/core/display.py:724: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")
